In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sksurv.metrics import brier_score
from sksurv.metrics import concordance_index_censored
from sksurv.nonparametric import kaplan_meier_estimator
from lifelines import KaplanMeierFitter
from sksurv.util import Surv
import wandb
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

### Data preprocessing

In [2]:
# Load data
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')
meta = pd.read_csv('Data/metaData.csv')

# Converting to dataframe
train = pd.DataFrame(train)
test = pd.DataFrame(test)

# Splitting train and test into X_train and y_train
X_train = train.drop(columns=['event', 'event_id', 'time_to_hit_hours'])
y_train = train[['time_to_hit_hours', 'event']]
y_train['event'] = y_train['event'].astype(int)

# y_train for gbsa
y_train_surv = Surv.from_arrays(event = y_train["event"].astype(bool), 
                                time = y_train["time_to_hit_hours"])
event_vals = y_train['event'].copy() # Later use for oof prediction loop
time_vals = y_train['time_to_hit_hours'].copy()

# Dropping event_id for X_test
X_test = test.drop(columns=['event_id'])
event_id = test['event_id'] # Save event_id for submitting

# Horizon
HORIZONS = [12, 24, 48, 72]
DROP_HORIZONS = [24, 48, 72]


In [3]:
# Helper function
def compute_brier_sc(time, event, preds, horizons):
    brier_sc = np.zeros(len(horizons))

    for i, horizon in enumerate(horizons):
        valid = ~((event == 0) & (time < horizon))
        y_true = ((event == 1) & (time <= horizon)).astype(int)
        brier_sc[i] = np.mean((y_true[valid] - preds[valid, i]) ** 2)

    return brier_sc

'''
Hàm dưới đây sẽ không được dùng để đánh giá

def compute_c_index(time, event, risk):
    n = len(event)
    concordant = total = 0
    for i in range(n):
        if event[i] == 0: # Skip censored event
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]: 
                continue
            total += 1
            if risk[i] == risk[j]:
                concordant += 0.5
            elif risk[i] > risk[j]:
                concordant += 1
    if total > 0:
        return concordant / total
    return 0.5
'''
def compute_hybrid_sc(time, event, preds, horizons):
    risk_score = preds[:, 1] * 0.3 + preds[:, 2] * 0.4 + preds[:, 3] * 0.3
    c_index = concordance_index_censored(
        event.astype(bool),
        time,
        risk_score
    )[0]
    brier_score = compute_brier_sc(time, event, preds, horizons)
    p24 = brier_score[1]
    p48 = brier_score[2]
    p72 = brier_score[3]
    weighted_brier = 0.3*p24 + 0.4*p48 + 0.3*p72
    hybrid_score = 0.3*c_index + 0.7*(1 - weighted_brier)
    return hybrid_score

### Gradient Boost Survival Analysis

In [4]:
# Total folds
N_FOLDS_GBSA = 5

# Initialize oof matrix prediction
oof_preds_gbsa = np.zeros((len(X_train), 4)) # 4 horizons: 12h, 24h, 48h, 72h

# Initialize final average test prediction
gbsa_test_preds = np.zeros((len(X_test), 4))

# Random seeds — 5 is enough for good variance reduction with 6 configs (30 total runs)
GBSA_SEEDS = [42, 526, 2026, 67, 478]

# GBSA configs — all Brier-optimized:
#   - Shallow trees (depth 2-3) to avoid overfitting 177-row folds
#   - Slow learning rate + many estimators for smoother survival curves
#   - Larger min_samples_leaf to suppress overconfident leaf outputs
#   - Subsample 0.7-0.85 for diversity across the ensemble
gbsa_configs = [
    # Conservative: deep ensemble, slow LR, big leaves
    {"learning_rate":0.005, "subsample":0.85, "max_depth":3, "min_samples_leaf":18, "min_samples_split":4, "n_estimators":2000},
    # Balanced default
    {"learning_rate":0.010, "subsample":0.80, "max_depth":3, "min_samples_leaf":15, "min_samples_split":3, "n_estimators":1200},
    # Shallow, more aggressive
    {"learning_rate":0.010, "subsample":0.75, "max_depth":2, "min_samples_leaf":12, "min_samples_split":3, "n_estimators":1500},
    # Slightly faster LR for variety
    {"learning_rate":0.015, "subsample":0.70, "max_depth":3, "min_samples_leaf":12, "min_samples_split":3, "n_estimators":900},
    # Very conservative
    {"learning_rate":0.005, "subsample":0.90, "max_depth":2, "min_samples_leaf":20, "min_samples_split":4, "n_estimators":2200},
    # Different subsample, slower LR
    {"learning_rate":0.008, "subsample":0.65, "max_depth":3, "min_samples_leaf":15, "min_samples_split":3, "n_estimators":1400},
]


##### Train GBSA

In [5]:
for config in gbsa_configs:
    for seed in GBSA_SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS_GBSA, shuffle=True, random_state=seed)
        seed_test_preds = np.zeros((len(X_test), len(HORIZONS)))

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, event_vals)):
            X_tr  = X_train.iloc[train_idx].copy()
            X_val = X_train.iloc[val_idx].copy()
            y_tr_surv  = y_train_surv[train_idx]

            # Initializing gbsa model
            gbsa = GradientBoostingSurvivalAnalysis(**config, random_state=seed)
            gbsa.fit(X_tr, y_tr_surv)

            # OOF
            val_preds_fold = np.zeros((len(val_idx), len(HORIZONS)))
            for i, sf in enumerate(gbsa.predict_survival_function(X_val)):
                for j, horizon in enumerate(HORIZONS):
                    val_preds_fold[i, j] = 1 - sf(min(horizon, sf.x[-1])) # type: ignore
            oof_preds_gbsa[val_idx] += val_preds_fold

            # Test predictions
            for i, sf in enumerate(gbsa.predict_survival_function(X_test)):
                for j, h in enumerate(HORIZONS):
                    seed_test_preds[i, j] += (1 - sf(min(h, sf.x[-1]))) # type: ignore

        # Average across folds for this (config, seed) pair
        gbsa_test_preds += seed_test_preds / N_FOLDS_GBSA

# Average across all (config, seed) pairs
total_runs = len(gbsa_configs) * len(GBSA_SEEDS)
oof_preds_gbsa  /= total_runs
gbsa_test_preds /= total_runs

oof_preds_gbsa = np.clip(oof_preds_gbsa, 0, 1)
gbsa_test_preds = np.clip(gbsa_test_preds, 0, 1)

In [6]:
risk_score = oof_preds_gbsa[:, 1] * 0.3 + oof_preds_gbsa[:, 2] * 0.4 + oof_preds_gbsa[:, 3] * 0.3
risk_score.shape

(221,)

### GBSA evaluation (c_index, brier score)

In [7]:
c_index = concordance_index_censored(
    event_vals.astype(bool),
    time_vals,
    risk_score
)[0]
brier_score = compute_brier_sc(time_vals, event_vals, oof_preds_gbsa, HORIZONS)
hybrid_sc = compute_hybrid_sc(time_vals, event_vals, oof_preds_gbsa, HORIZONS)

print(f"C_index: {c_index:.4f}\n"
      f"Brier p24h: {brier_score[1]:.4f}\n"
      f"Brier p48h: {brier_score[2]:.4f}\n"
      f"Brier p72h: {brier_score[3]:.4f}\n" # Dell care
      f"Hybrid score: {hybrid_sc:.4f}\n")

C_index: 0.9424
Brier p24h: 0.0286
Brier p48h: 0.0161
Brier p72h: 0.0010
Hybrid score: 0.9720



### Train lgbm (12 24 48)

In [8]:
def create_features(df):
    result = df.copy()
    dist = result['dist_min_ci_0_5h'].clip(lower=1)
    speed = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']
    result['log_distance'] = np.log1p(dist)
    result['inv_distance'] = 1 / (dist / 1000 + 0.1)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance'] = np.sqrt(dist)
    result['dist_km'] = dist / 1000
    result['dist_km_sq'] = (dist / 1000) ** 2
    result['dist_rank'] = dist.rank(pct=True)
    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio'] = area_first / (dist / 1000 + 0.1)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)
    result['has_movement'] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result['eta_hours'] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result['log_eta'] = np.log1p(result['eta_hours'].clip(0, 9999))
    radial_growth = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result['threat_score'] = result['alignment_abs'] * speed / np.log1p(dist)
    result['fire_urgency'] = perimeters * speed
    result['growth_intensity'] = result['area_growth_rate_ha_per_h'] * perimeters
    result['zone_critical'] = (dist < 5000).astype(float)
    result['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result['zone_safe'] = (dist >= 10000).astype(float)
    result['is_summer'] = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) & (result['event_start_hour'] < 20)).astype(float)
    drop_cols = ['relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
                 'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h']
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result


X_train = create_features(X_train)
X_test  = create_features(X_test)
X_train.head(3)

,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,log_area_ratio_0_5h,radial_growth_m,...,effective_closing_speed,eta_effective,threat_score,fire_urgency,growth_intensity,zone_critical,zone_warning,zone_safe,is_summer,is_afternoon
0,3,4.265188,0,79.696304,0.036086,0.674281,4.390693,1.354787,0.03545,9.007182,...,2.11179,2919.855353,-0.000639,-0.306002,2.022842,0.0,1.0,0.0,0.0,1.0
1,2,1.169918,0,8.946749,0.000000,0.000000,2.297246,0.000000,0.00000,0.000000,...,0.00000,9999.000000,0.000000,0.000000,0.000000,1.0,0.0,0.0,1.0,0.0
2,4,4.777526,0,106.482638,0.000000,0.000000,4.677329,0.000000,0.00000,0.000000,...,0.00000,9999.000000,0.000000,0.000000,0.000000,1.0,0.0,0.0,1.0,0.0


In [9]:
# Initialize N_folds for lgbm
N_FOLDS_LGBM = 5

# Initialize random seeds for kFOLD
LGBM_SEEDS = [526, 484, 749, 852, 848, 123, 456, 789, 147, 369, 165, 986, 8945, 8946, 592]

# Horizon
HORIZONS = [12, 24, 48, 72]
DROP_HORIZONS_72 = [12, 24, 48]

# Initialize oof matrix prediction
oof_preds_lgbm = np.zeros((len(X_train), 3))

# Initialize final average test prediction
lgbm_test_preds = np.zeros((len(X_test), 3))

# For safety
X_train_lgbm = X_train.values
X_test_lgbm = X_test.values

# Brier-optimized LightGBM configs.
# 12h: events very rare → heavy regularization, shallow trees, large min_child_samples
# 24h: moderate events → balance discrimination and calibration
# 48h: most events occurred → can afford slightly more capacity, lighter reg
lgb_cfgs = {
    12: [
        # Heavy regularization, shallow, large leaves — avoid overconfident predictions
        {"max_depth":2, "learning_rate":0.02, "n_estimators":400, "subsample":0.7,  "colsample_bytree":0.7,  "min_child_samples":15, "reg_alpha":2.0, "reg_lambda":5.0, "num_leaves":4},
        {"max_depth":2, "learning_rate":0.01, "n_estimators":600, "subsample":0.6,  "colsample_bytree":0.6,  "min_child_samples":18, "reg_alpha":3.0, "reg_lambda":6.0, "num_leaves":3},
        {"max_depth":2, "learning_rate":0.03, "n_estimators":300, "subsample":0.75, "colsample_bytree":0.75, "min_child_samples":12, "reg_alpha":1.5, "reg_lambda":4.0, "num_leaves":4},
        {"max_depth":3, "learning_rate":0.015,"n_estimators":500, "subsample":0.7,  "colsample_bytree":0.7,  "min_child_samples":15, "reg_alpha":2.0, "reg_lambda":4.0, "num_leaves":5},
        {"max_depth":2, "learning_rate":0.025,"n_estimators":350, "subsample":0.8,  "colsample_bytree":0.65, "min_child_samples":20, "reg_alpha":2.5, "reg_lambda":3.0, "num_leaves":4},
    ],
    24: [
        # Moderate depth, moderate reg, slow learning rate for calibration
        {"max_depth":3, "learning_rate":0.02, "n_estimators":400, "subsample":0.75, "colsample_bytree":0.75, "min_child_samples":10, "reg_alpha":0.5, "reg_lambda":2.0, "num_leaves":7},
        {"max_depth":3, "learning_rate":0.015,"n_estimators":500, "subsample":0.8,  "colsample_bytree":0.8,  "min_child_samples":10, "reg_alpha":0.3, "reg_lambda":1.5, "num_leaves":6},
        {"max_depth":2, "learning_rate":0.025,"n_estimators":350, "subsample":0.7,  "colsample_bytree":0.7,  "min_child_samples":12, "reg_alpha":0.8, "reg_lambda":2.5, "num_leaves":4},
        {"max_depth":3, "learning_rate":0.03, "n_estimators":300, "subsample":0.7,  "colsample_bytree":0.7,  "min_child_samples":8,  "reg_alpha":0.5, "reg_lambda":2.0, "num_leaves":7},
        {"max_depth":3, "learning_rate":0.01, "n_estimators":700, "subsample":0.85, "colsample_bytree":0.75, "min_child_samples":10, "reg_alpha":0.4, "reg_lambda":1.8, "num_leaves":6},
    ],
    48: [
        # Lighter reg, smaller min_child_samples — more events to learn from
        {"max_depth":3, "learning_rate":0.025,"n_estimators":350, "subsample":0.8,  "colsample_bytree":0.8,  "min_child_samples":6,  "reg_alpha":0.1, "reg_lambda":1.0, "num_leaves":6},
        {"max_depth":2, "learning_rate":0.04, "n_estimators":250, "subsample":0.8,  "colsample_bytree":0.8,  "min_child_samples":5,  "reg_alpha":0.1, "reg_lambda":1.0, "num_leaves":6},
        {"max_depth":3, "learning_rate":0.02, "n_estimators":450, "subsample":0.85, "colsample_bytree":0.85, "min_child_samples":6,  "reg_alpha":0.05,"reg_lambda":0.5, "num_leaves":7},
        {"max_depth":2, "learning_rate":0.03, "n_estimators":350, "subsample":0.75, "colsample_bytree":0.75, "min_child_samples":8,  "reg_alpha":0.2, "reg_lambda":0.8, "num_leaves":5},
        {"max_depth":3, "learning_rate":0.015,"n_estimators":600, "subsample":0.85, "colsample_bytree":0.8,  "min_child_samples":6,  "reg_alpha":0.1, "reg_lambda":0.8, "num_leaves":6},
    ],
}


def compute_ipcw(times, events, horizon):
    """Kaplan-Meier-based IPCW weights for censoring-corrected binary labels."""
    unique_t = np.sort(np.unique(times))
    surv     = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk        = (times >= t).sum()
        censored_at_t  = ((times == t) & (events == 0)).sum()
        if at_risk > 0:
            surv[i] = 1 - censored_at_t / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]

    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0

    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
    return weights


###### ai rảnh thì refactor cái này giùm t tks you<3

In [10]:
time_vals = train["time_to_hit_hours"].to_numpy()
event_vals = train["event"].to_numpy()
for i, horizon in enumerate(DROP_HORIZONS_72):    

    y_train_lgbm = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    uncensored_row = ~((event_vals == 0) & (time_vals < horizon))
    uncensored_idx = np.where(uncensored_row)[0]
    censored_idx = np.where(~uncensored_row)[0] # Fill in later with predict_proba
    horizon_oof = np.zeros(len(X_train_lgbm))
    test_oof = np.zeros(len(X_test_lgbm))

    for config in lgb_cfgs[horizon]:
        for seed in LGBM_SEEDS:
            skf = StratifiedKFold(n_splits=N_FOLDS_LGBM, shuffle=True, random_state=seed)
            seed_oof_preds = np.zeros(len(X_train_lgbm))
            censored_pred_sum = np.zeros(len(censored_idx))

            for fold, (train_idx, val_idx) in enumerate(skf.split(uncensored_idx, y_train_lgbm[uncensored_row])):
                valid_train_idx = uncensored_idx[train_idx] # Hết biết đặt tên gì r
                valid_val_idx = uncensored_idx[val_idx]

                X_tr = X_train_lgbm[valid_train_idx].copy()
                X_val = X_train_lgbm[valid_val_idx].copy()
                y_tr = y_train_lgbm[valid_train_idx].copy()

                # Calculate the IPCW
                ipcw_weights = compute_ipcw(time_vals[valid_train_idx], event_vals[valid_train_idx], horizon)

                lgbm = LGBMClassifier(**config, random_state=seed, verbose=-1)
                lgbm.fit(
                    X_tr, y_tr,
                    sample_weight=ipcw_weights
                )
                seed_oof_preds[valid_val_idx] = lgbm.predict_proba(X_val)[:, 1] # type: ignore
                if (len(censored_idx) > 0):
                    censored_pred_sum += lgbm.predict_proba(X_train_lgbm[censored_idx])[:, 1] # type: ignore         
            if len(censored_idx) > 0:
                seed_oof_preds[censored_idx] = censored_pred_sum / N_FOLDS_LGBM
            horizon_oof += seed_oof_preds

            ipcw_full = compute_ipcw(time_vals, event_vals, horizon) # all of the prediction is filled
            lgbm_full = LGBMClassifier(**config, random_state=seed, verbose=-1)
            lgbm_full.fit(
                X_train_lgbm,
                y_train_lgbm,
                sample_weight=ipcw_full
            )

            test_oof += lgbm_full.predict_proba(X_test_lgbm)[:, 1] # type: ignore

    oof_preds_lgbm[:, i] = horizon_oof / (len(LGBM_SEEDS) * len(lgb_cfgs[horizon]))
    lgbm_test_preds[:, i] = test_oof / (len(LGBM_SEEDS) * len(lgb_cfgs[horizon]))

### LGBM Evaluation

In [11]:
brier_score = compute_brier_sc(time_vals, event_vals, oof_preds_lgbm, DROP_HORIZONS_72)
print(f"Brier p12h: {brier_score[0]:.4f}\n" 
      f"Brier p24h: {brier_score[1]:.4f}\n"
      f"Brier p48h: {brier_score[2]:.4f}\n") 

Brier p12h: 0.0545
Brier p24h: 0.0259
Brier p48h: 0.0130



In [12]:
# Power calibration — disabled. The blend grid search optimizes raw probabilities
# directly, so post-hoc rescaling here can fight the search. Re-enable to A/B test:
# oof_preds_lgbm[:, 2] = (oof_preds_lgbm[:, 2]) ** 1.1
# lgbm_test_preds[:, 2] = (lgbm_test_preds[:, 2]) ** 1.1


### Blend weight

In [13]:
# ---- Auto-search blend weights with hard distance rule on OOF ----
# 
# DATA INSIGHT: dist_min_ci_0_5h is a near-perfect predictor of `event` in this dataset:
#   - dist <  5000m (n=69): ALL 69 hit (100%)
#   - dist >= 5000m (n=152): ZERO hit (0%)
# So the Brier-optimal prediction for far rows is exactly 0.0 at every horizon.
# Any positive prediction on far rows is pure squared-error tax.
#
# We therefore (a) zero far rows during eval and submission, and (b) grid-search
# blend weights to maximize hybrid score WITH the rule applied — these two
# work together: zeroing far rows changes which weights are optimal on near rows.

W_GBSA_12, W_LGB_12 = 0.97, 0.03   # 12h is GBSA-friendly, leave alone

# Build masks from raw distance (most reliable)
train_far_mask = (train["dist_min_ci_0_5h"].to_numpy() >= 5000)
test_far_mask  = (test ["dist_min_ci_0_5h"].to_numpy() >= 5000)
print(f"Train far rows: {train_far_mask.sum()}/{len(train_far_mask)}")
print(f"Test  far rows: {test_far_mask.sum()}/{len(test_far_mask)}")


def _eval_blend(w24, w48, apply_rule=True):
    """Build candidate OOF blend, optionally apply far-zero rule, return hybrid + briers."""
    cand = np.zeros((len(oof_preds_lgbm), 4))
    cand[:, 0] = oof_preds_gbsa[:, 0] * W_GBSA_12 + oof_preds_lgbm[:, 0] * W_LGB_12
    cand[:, 1] = oof_preds_gbsa[:, 1] * w24       + oof_preds_lgbm[:, 1] * (1 - w24)
    cand[:, 2] = oof_preds_gbsa[:, 2] * w48       + oof_preds_lgbm[:, 2] * (1 - w48)
    cand[:, 3] = 1.0
    cand = np.maximum.accumulate(cand, axis=1)  # monotonicity
    if apply_rule:
        cand[train_far_mask, 0] = 0.0
        cand[train_far_mask, 1] = 0.0
        cand[train_far_mask, 2] = 0.0
        # leave p72 alone — no far row is evaluable at 72h anyway, but C-index uses it
    h = compute_hybrid_sc(time_vals, event_vals, cand, HORIZONS)
    b = compute_brier_sc(time_vals, event_vals, cand, HORIZONS)
    return h, b[1], b[2]


# Diagnostic: original blend, with and without rule
o_h_norule, o_b24_n, o_b48_n = _eval_blend(0.95, 0.70, apply_rule=False)
o_h_rule,   o_b24_r, o_b48_r = _eval_blend(0.95, 0.70, apply_rule=True)
print(f"\nOriginal weights (w24=0.95, w48=0.70):")
print(f"  no rule    : hybrid={o_h_norule:.4f}  b24={o_b24_n:.4f}  b48={o_b48_n:.4f}")
print(f"  far-zero   : hybrid={o_h_rule:.4f}  b24={o_b24_r:.4f}  b48={o_b48_r:.4f}")

# Grid search WITH the rule applied (since rule will be in final submission)
grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)
best = (-np.inf, None, None, None, None)
for w24 in grid:
    for w48 in grid:
        h, b24, b48 = _eval_blend(w24, w48, apply_rule=True)
        if h > best[0]:
            best = (h, w24, w48, b24, b48)

best_h, best_w24, best_w48, best_b24, best_b48 = best
print(f"\nBest blend (with far-zero rule): w24={best_w24:.2f}, w48={best_w48:.2f}")
print(f"  hybrid={best_h:.4f}  b24={best_b24:.4f}  b48={best_b48:.4f}")
print(f"  gain over original (with rule): hybrid +{best_h - o_h_rule:.4f}")
print(f"  gain over original (no rule)  : hybrid +{best_h - o_h_norule:.4f}")

# Set final weights for downstream cells
W_GBSA_24, W_LGB_24 = best_w24, 1 - best_w24
W_GBSA_48, W_LGB_48 = best_w48, 1 - best_w48


Train far rows: 152/221
Test  far rows: 67/95

Original weights (w24=0.95, w48=0.70):
  no rule    : hybrid=0.9733  b24=0.0284  b48=0.0139
  far-zero   : hybrid=0.9737  b24=0.0275  b48=0.0131

Best blend (with far-zero rule): w24=0.95, w48=0.00
  hybrid=0.9738  b24=0.0275  b48=0.0128
  gain over original (with rule): hybrid +0.0001
  gain over original (no rule)  : hybrid +0.0005


In [14]:
blended_oof = np.zeros((len(oof_preds_lgbm), 4))

blended_oof[:, 0] = oof_preds_gbsa[:, 0] * W_GBSA_12 + oof_preds_lgbm[:, 0] * W_LGB_12
blended_oof[:, 1] = oof_preds_gbsa[:, 1] * W_GBSA_24 + oof_preds_lgbm[:, 1] * W_LGB_24
blended_oof[:, 2] = oof_preds_gbsa[:, 2] * W_GBSA_48 + oof_preds_lgbm[:, 2] * W_LGB_48
blended_oof[:, 3] = 1.0
blended_oof = np.maximum.accumulate(blended_oof, axis=1) # type: ignore

# Hard distance rule: dist >= 5000m → 0% hit probability at 12/24/48h.
# Justified by data: 0/152 far-row train events. Pure noise removal.
blended_oof[train_far_mask, 0] = 0.0
blended_oof[train_far_mask, 1] = 0.0
blended_oof[train_far_mask, 2] = 0.0

blended_oof_copy = pd.DataFrame(blended_oof.copy())

brier_score = compute_brier_sc(time_vals, event_vals, blended_oof, HORIZONS)
hybrid_sc   = compute_hybrid_sc(time_vals, event_vals, blended_oof, HORIZONS)
print(f"FINAL OOF (after rule):\n"
      f"  Brier p24h : {brier_score[1]:.4f}\n"
      f"  Brier p48h : {brier_score[2]:.4f}\n"
      f"  Brier p72h : {brier_score[3]:.4f}\n"
      f"  Hybrid     : {hybrid_sc:.4f}")


FINAL OOF (after rule):
  Brier p24h : 0.0275
  Brier p48h : 0.0128
  Brier p72h : 0.0000
  Hybrid     : 0.9738


In [15]:
final_preds = np.zeros((len(gbsa_test_preds), 4))

final_preds[:, 0] = gbsa_test_preds[:, 0] * W_GBSA_12 + lgbm_test_preds[:, 0] * W_LGB_12
final_preds[:, 1] = gbsa_test_preds[:, 1] * W_GBSA_24 + lgbm_test_preds[:, 1] * W_LGB_24
final_preds[:, 2] = gbsa_test_preds[:, 2] * W_GBSA_48 + lgbm_test_preds[:, 2] * W_LGB_48
final_preds[:, 3] = gbsa_test_preds[:, 3]
final_preds = np.maximum.accumulate(final_preds, axis=1) # type: ignore

# Apply hard distance rule on test set too (uses test_far_mask defined above).
# Test set has the same gap: near rows max at 4839m, far rows start at 5787m.
final_preds[test_far_mask, 0] = 0.0
final_preds[test_far_mask, 1] = 0.0
final_preds[test_far_mask, 2] = 0.0

final_preds_copy = pd.DataFrame(final_preds.copy())
final_preds_copy.head(3)


,0,1,2,3
0,0.000000,0.000000,0.000000,0.095923
1,0.693031,0.939206,0.996909,0.996909
2,0.000000,0.000000,0.000000,0.095424


### Final submissions 

In [16]:
# p72 logic on test:
#   near rows (dist < 5km): all 69/69 train near rows hit by 72h → set p72 = 1.0
#   far  rows (dist >=5km): 0/152 train far rows hit within 72h → set p72 = 0.0
final_preds[~test_far_mask, 3] = 1.0
final_preds[ test_far_mask, 3] = 0.0
# Final monotonicity guard
final_preds = np.maximum.accumulate(final_preds, axis=1)

submission = {
    'event_id' : event_id,
    'prob_12h' : final_preds[:, 0],# type: ignore
    'prob_24h' : final_preds[:, 1],# type: ignore
    'prob_48h' : final_preds[:, 2],# type: ignore
    'prob_72h' : final_preds[:, 3] # type: ignore
}
submission = pd.DataFrame(submission)
submission.to_csv('final_preds.csv', index=False)
